1. Mô tả các thuật toán thực tế đã chọn
Trong đồ án này, tôi ưu tiên sử dụng các thuật toán dựa trên cấu trúc cây (Tree-based models) vì khả năng xử lý xuất sắc dữ liệu dạng bảng và các mối quan hệ phi tuyến phức tạp.

1.1. XGBoost (Extreme Gradient Boosting)
Nguyên lý: Đây là một thuật toán thuộc họ Gradient Boosting Decision Tree (GBDT). XGBoost xây dựng một chuỗi các cây quyết định tuần tự, trong đó mỗi cây mới được huấn luyện để dự đoán sai số (residuals) của các cây trước đó.

Lý do chọn: * Hiệu năng cao: Có khả năng xử lý song song, giúp rút ngắn thời gian huấn luyện trên tập dữ liệu hàng triệu bản ghi.

Chống Overfitting: Tích hợp sẵn các kỹ thuật hiệu chỉnh (L1, L2 Regularization).

Xử lý dữ liệu nhiễu: Cực kỳ hiệu quả với các biến có giá trị ngoại lai (outliers) như tọa độ hoặc thời gian di chuyển bất thường.
2. Mô tả các thuộc tính và đặc trưng (Final Features)
Để mô hình đạt độ chính xác cao nhất (giảm chỉ số RMSLE), dữ liệu thô đã được biến đổi thành bộ đặc trưng (features) cuối cùng sau đây:

2.1. Nhóm Đặc trưng Thời gian (Temporal Features)
Giúp mô hình bắt kịp quy luật giao thông theo chu kỳ của thành phố New York:

pickup_hour: Giờ đón khách (0-23). Xác định các khung giờ cao điểm và giờ thấp điểm.

day_of_week: Thứ trong tuần (Thứ 2 - Chủ Nhật). Phản ánh sự khác biệt giữa lưu lượng giao thông ngày đi làm và ngày nghỉ.

month: Tháng trong năm (1-12). Ảnh hưởng bởi yếu tố mùa và thời tiết.

2.2. Nhóm Đặc trưng Không gian & Khoảng cách (Spatial Features)
Đây là nhóm biến quan trọng nhất để xác định thời gian di chuyển:

dist_haversine: Khoảng cách đường chim bay giữa hai tọa độ (vĩ độ/kinh độ).

dist_manhattan: Khoảng cách di chuyển theo lưới đường bộ (tổng khoảng cách theo trục ngang và dọc), cực kỳ phù hợp với cấu trúc đường phố bàn cờ của New York.

bearing (Hướng di chuyển): Góc di chuyển giữa điểm đón và điểm trả, giúp mô hình phân biệt hướng đi vào trung tâm hay đi ra ngoại ô.

2.3. Nhóm Đặc trưng Cụm địa lý (Clustering Features)
pickup_cluster / dropoff_cluster: Sử dụng thuật toán K-Means để gom nhóm hàng triệu tọa độ rời rạc thành các cụm khu vực (ví dụ: khu vực Sân bay JFK, khu vực Manhattan, khu vực Brooklyn). Điều này giúp mô hình hiểu được đặc thù giao thông của từng vùng địa lý cụ thể.

2.4. Nhóm Đặc trưng Khác
passenger_count: Số lượng hành khách trên xe.

vendor_id: Mã nhà cung cấp dịch vụ vận tải.

store_and_fwd_flag: Trạng thái lưu trữ và gửi dữ liệu của thiết bị định vị trên xe.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('nyc-taxi-trip-duration')

print("Path to competition files:", path)

100%|██████████| 85.8M/85.8M [00:05<00:00, 17.9MB/s]

Extracting files...


Path to competition files: C:\Users\trant\.cache\kagglehub\competitions\nyc-taxi-trip-duration


In [4]:
import zipfile
import os

os.makedirs("data", exist_ok=True)

for file in os.listdir():
    if file.endswith(".zip"):
        print("Extracting:", file)
        with zipfile.ZipFile(file, 'r') as zip_ref:
            zip_ref.extractall("data")

print("All files extracted!")

Extracting: sample_submission.zip
Extracting: test.zip
Extracting: train.zip
All files extracted!


1. Đầu tiên là đọc dữ liệu

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

#Đọc dữ liệu
df = pd.read_csv('data/train.csv')


2. Chuyển đổi giá trị (Data Transformation)

In [3]:
# Chuyển đổi định dạng datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Chuyển đổi biến mục tiêu sang Log-scale để giảm độ lệch (Skewness)
df['trip_duration_log'] = np.log1p(df['trip_duration'])

# Chuyển đổi biến phân loại store_and_fwd_flag sang số (Y=1, N=0)
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y': 1, 'N': 0})

3. Trích xuất thuộc tính (Feature Extraction)

In [4]:
# A. Nhóm đặc trưng thời gian
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.weekday
df['month'] = df['pickup_datetime'].dt.month
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# B. Nhóm đặc trưng khoảng cách
def haversine_array(lat1, lng1, lat2, lng2):
    # Tính khoảng cách đường chim bay (Haversine)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    AVG_EARTH_RADIUS = 6371  # km
    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng * 0.5) ** 2
    return 2 * AVG_EARTH_RADIUS * np.arcsin(np.sqrt(d))

def manhattan_distance(lat1, lng1, lat2, lng2):
    # Tính khoảng cách Manhattan
    a = haversine_array(lat1, lng1, lat1, lng2)
    b = haversine_array(lat1, lng1, lat2, lng1)
    return a + b

df['dist_haversine'] = haversine_array(df['pickup_latitude'], df['pickup_longitude'], 
                                        df['dropoff_latitude'], df['dropoff_longitude'])

df['dist_manhattan'] = manhattan_distance(df['pickup_latitude'], df['pickup_longitude'], 
                                           df['dropoff_latitude'], df['dropoff_longitude'])

# C. Tính hướng di chuyển (Bearing)
def bearing_array(lat1, lng1, lat2, lng2):
    lng_delta_rad = np.radians(lng2 - lng1)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    return np.degrees(np.atan2(y, x))

df['direction'] = bearing_array(df['pickup_latitude'], df['pickup_longitude'], 
                                df['dropoff_latitude'], df['dropoff_longitude'])

4. Gộp bảng / làm sạch (Cleaning & Merging)

In [5]:
# Loại bỏ các Outliers rõ rệt để mô hình không bị nhiễu
# Ví dụ: Chuyến đi > 24h (86400s) hoặc < 1 phút (60s)
df = df[(df['trip_duration'] > 60) & (df['trip_duration'] < 86400)]
# Kiểm tra lại 5 dòng đầu tiên
print(df[['hour', 'dist_manhattan', 'direction', 'trip_duration_log']].head())

   hour  dist_manhattan   direction  trip_duration_log
0    17        1.735433   99.970196           6.122493
1     0        2.430506 -117.153768           6.498282
2    11        8.203575 -159.680165           7.661527
3    19        1.661331 -172.737700           6.063785
4    13        1.199457  179.473585           6.077642
